# Qian: BitFit linear/cosine heads and crop/flip validation

Train two fresh BitFit models on the same 1,000-class split (40 fitting + 5 holdout images/class), three epochs each. Only the head design changes. Backbone biases and head LR are 1e-3; zero weight decay, seed 42, prototype initialization from fitting images only. Cosine scale starts at 20. The classifier remains 10,000-way.

Select each arm's `best.pt` using standard crop 0.9 without flip during training. Evaluate that fixed checkpoint on the same labeled holdout at crop fractions 0.9, 0.7, 0.75 and 0.8, each with/without horizontal flip: 16 local validation results, no submission files. A100 recommended. Use Runtime → Run all.

Three epochs are a pilot, not proof of convergence. This holdout is used for selection; confirm the winner on another seed or larger split before the final all-data run.


In [1]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_bitfit_ablation"
TEAM = "08"
CONFIG = "configs/bitfit_head_ablation_336.yaml"
EXPERIMENT_NAME = "qq_bitfit_heads_crop_336_seed42_v1"
EXPERIMENT_DIR = f"{DRIVE_OUT}/{EXPERIMENT_NAME}"
SPLIT_DIR = f"/content/{EXPERIMENT_NAME}_splits"
# Use a new name if changing the recipe or epoch count.


## 1. Clone or pull the repo

In [2]:
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


GitHub PAT (hidden; repo scope). Leave empty for a public repo: ··········
Branch: qq_bitfit_ablation
Commit: aaa14da


## 2. Install requirements

In [3]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [4]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

Mounted at /content/drive
Writing runs to /content/drive/.shortcut-targets-by-id/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR/CS604_RUNS
Existing items: ['full_ft_20261005_074553_id962394', 'full_ft_20261006_031830', 'headinit_full_ft_336_20261006_110343', 'prototypes_336.pt', 'prototypes_336.json', 'full_ft_20261006_122039', 'lora_20261007_051926', 'qq_selected_layer_4bae5d64c9c7ecfb', 'last_blocks_20261008_123344_id969549', 'qq_prefix_ablation_336_seed42_v1', 'pilot_adapter_336_20261009_125653', 'adapter_20261009_131150', 'prototypes_224.json', 'prototypes_224.pt', 'last_blocks_20261010_022115', 'val_team_Wenqi.txt', 'prefix_20261010_000404_id974631', 'qq_bitfit_ablation_336_subset_seed42_v2', 'prototypes_336.usage_metadata.json', 'bitfit_20261011_024724_id976355']


## 4. Download data to `/content/data`

In [5]:
subprocess.run(["python", "scripts/download_data.py", "--dest", DATA_DIR], check=True)


CompletedProcess(args=['python', 'scripts/download_data.py', '--dest', '/content/data'], returncode=0)

## 5. Linear versus cosine head

Fresh pretrained initialization for both arms. Fit-only prototypes are shared; never use the all-data prototypes for this holdout. A new experiment directory prevents reusing old results after code changes. `best.pt` uses highest holdout accuracy; `last.pt` is the final epoch. If both curves are still improving at epoch 3, extend both equally using a new experiment name.


In [8]:
import torch
from src.utils import load_config, apply_overrides
from src.peft.bitfit_ablation import prepare, stage_config, ensure_prototypes, run_arm, write_results
import pandas as pd

assert torch.cuda.is_available(), "Select an A100 GPU runtime."
BASE = load_config(CONFIG)
apply_overrides(BASE, data_root=DATA_DIR, output_dir=EXPERIMENT_DIR)
EXPERIMENT = Path(EXPERIMENT_DIR)
EXPERIMENT.mkdir(parents=True, exist_ok=True)
ROOTS = prepare(BASE, EXPERIMENT, split_dir=SPLIT_DIR)
results = []
for head in ("linear", "cosine"):
    arm = {"arm": f"all_biases_{head}", "method": "bitfit", "bitfit_scope": "all", "head_type": head}
    cfg = stage_config(BASE, EXPERIMENT, ROOTS["pilot"], "head", arm, 42, 1e-3)
    ensure_prototypes(cfg, allow_missing_classes=True)
    result = run_arm(cfg)
    results.append({**result, "head_type": head})
write_results(EXPERIMENT, "head", results)
display(pd.DataFrame(results))
for result in results:
    subprocess.run(["python", "scripts/plot_curves.py", "--run_dir", result["run_dir"],
                    "--out_dir", str(Path(result["run_dir"]) / "figures")], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
(EXPERIMENT / "launcher_commit.txt").write_text(commit + "\n")


[ablation] reusing /content/drive/MyDrive/CS604_RUNS/qq_bitfit_heads_crop_336_seed42_v1/runs/head/all_biases_linear/seed42
[ablation] reusing /content/drive/MyDrive/CS604_RUNS/qq_bitfit_heads_crop_336_seed42_v1/runs/head/all_biases_cosine/seed42


,adaptation_parameters,adaptation_pct,arm,before_val_acc,best_epoch,best_val_acc,bitfit_scope,early_dip,final_val_acc,final_val_loss,...,original_backbone_parameters,peak_vram_gb,run_dir,seed,timm_version,torch_version,training_images_per_second,training_minutes,wall_gpu_hours,head_type
0,102912,0.119683,all_biases_linear,0.6896,3,0.8792,all,False,0.8792,0.496490,...,85986816,7.012053,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,42,1.0.30,2.11.0+cu130,498.359566,4.01,0.074148,linear
1,102912,0.119683,all_biases_cosine,0.6902,3,0.8904,all,False,0.8904,0.450855,...,85986816,7.012026,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,42,1.0.30,2.11.0+cu130,494.559842,4.04,0.074693,cosine


41

## 6. Evaluate both best checkpoints with eight crop/flip settings

Accuracy is measured over the labeled holdout with the full 10,000-way classifier. Flip combines softmax probabilities from the original and mirrored images. Also record negative log likelihood of the resulting distribution, which is comparable across inference settings. The 0.9/no-flip result should reproduce the training-selected best accuracy, apart from numerical variation.


In [9]:
import gc
import math
from torch.utils.data import DataLoader
from src.data import ImageListDataset, build_transforms
from src.model import build_model
from src.predict import load_checkpoint
from src.utils import get_device, set_seed

@torch.no_grad()
def score_holdout(model, cfg, crop, flip, device):
    _, transform = build_transforms(cfg, eval_crop_pct=crop)
    dataset = ImageListDataset(cfg.data_root, "val", transform=transform)
    if not dataset.has_labels:
        raise ValueError("This sweep requires a labeled internal holdout.")
    loader = DataLoader(dataset, batch_size=int(cfg.batch_size), shuffle=False,
                        num_workers=int(cfg.num_workers), pin_memory=device.type == "cuda")
    correct, count, nll = 0, 0, 0.0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=bool(cfg.bf16) and device.type == "cuda"):
            logits = model(images)
            mirrored = model(torch.flip(images, dims=[3])) if flip else None
        log_probs = logits.float().log_softmax(dim=1)
        if flip:
            log_probs = torch.logaddexp(log_probs, mirrored.float().log_softmax(dim=1)) - math.log(2)
        correct += int((log_probs.argmax(1) == labels).sum())
        count += labels.numel()
        nll += float(torch.nn.functional.nll_loss(log_probs, labels, reduction="sum"))
    return {"val_acc": correct / count, "val_nll": nll / count, "holdout_images": count}

rows = []
device = get_device()
for result in results:
    run_dir = Path(result["run_dir"])
    checkpoint = run_dir / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    cfg = load_config(run_dir / "config.yaml")
    # Reattach current-session image symlinks; persisted manifests use the same split.
    cfg.data_root = str(ROOTS["pilot"])
    set_seed(int(cfg.seed))
    model = build_model(cfg).to(device)
    load_checkpoint(model, checkpoint, device)
    model.eval()
    for crop in (0.9, 0.7, 0.75, 0.8):
        for flip in (False, True):
            metrics = score_holdout(model, cfg, crop, flip, device)
            rows.append({"head_type": result["head_type"], "crop_pct": crop, "tta_flip": flip,
                         "best_epoch": result["best_epoch"], "checkpoint": str(checkpoint), **metrics})
            print(rows[-1])
            if crop == 0.9 and not flip and abs(metrics["val_acc"] - result["best_val_acc"]) > 0.001:
                raise RuntimeError("Baseline differs from recorded best accuracy; check the split/checkpoint.")
            pd.DataFrame(rows).to_csv(EXPERIMENT / "holdout_crop_flip_results.csv", index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
table = pd.DataFrame(rows)
display(table.sort_values("val_acc", ascending=False))
display(table.pivot(index="crop_pct", columns=["head_type", "tta_flip"], values="val_acc"))


[bitfit] scope=all: unfroze 74 backbone bias tensors; classifier also trainable
[predict] missing keys (76): ['cls_token', 'pos_embed', 'patch_embed.proj.weight', 'blocks.0.norm1.weight', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.proj.weight', 'blocks.0.norm2.weight', 'blocks.0.mlp.fc1.weight']...
{'head_type': 'linear', 'crop_pct': 0.9, 'tta_flip': False, 'best_epoch': 3, 'checkpoint': '/content/drive/MyDrive/CS604_RUNS/qq_bitfit_heads_crop_336_seed42_v1/runs/head/all_biases_linear/seed42/bitfit_20261014_121632/best.pt', 'val_acc': 0.8792, 'val_nll': 0.49652825808525086, 'holdout_images': 5000}
{'head_type': 'linear', 'crop_pct': 0.9, 'tta_flip': True, 'best_epoch': 3, 'checkpoint': '/content/drive/MyDrive/CS604_RUNS/qq_bitfit_heads_crop_336_seed42_v1/runs/head/all_biases_linear/seed42/bitfit_20261014_121632/best.pt', 'val_acc': 0.88, 'val_nll': 0.49054100914001464, 'holdout_images': 5000}
{'head_type': 'linear', 'crop_pct': 0.7, 'tta_flip': False, 'best_epoch': 3, 'checkpoint': '/co

,head_type,crop_pct,tta_flip,best_epoch,checkpoint,val_acc,val_nll,holdout_images
13,cosine,0.75,True,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8980,0.424829,5000
11,cosine,0.70,True,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8962,0.432183,5000
15,cosine,0.80,True,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8962,0.429491,5000
12,cosine,0.75,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8956,0.427935,5000
14,cosine,0.80,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8944,0.433193,5000
10,cosine,0.70,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8938,0.436303,5000
8,cosine,0.90,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8904,0.450855,5000
9,cosine,0.90,True,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8886,0.447533,5000
4,linear,0.75,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8852,0.465620,5000
2,linear,0.70,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_he...,0.8840,0.469949,5000


head_type  linear          cosine        
tta_flip    False   True    False   True 
crop_pct                                 
0.70       0.8840  0.8840  0.8938  0.8962
0.75       0.8852  0.8840  0.8956  0.8980
0.80       0.8838  0.8836  0.8944  0.8962
0.90       0.8792  0.8800  0.8904  0.8886